## 5. Convert CSV to PARQUET File

Code learned and followed from AAI 540 Lab 2

In [46]:
import boto3
import sagemaker
import pandas as pd

from pyathena import connect
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

In [55]:
# Set S3 path to Parquet data
s3_path_parquet = f's3://{bucket}/CCPP/data//parquet'
# Set Athena parameters
# Set Athena parameters
database_name = "ccpp_aws_fp"
table_name_csv = "final_data_csv"
table_name_parquet = "final_data_parquet_v1"

In [56]:
%store s3_path_parquet

Stored 's3_path_parquet' (str)


In [64]:
!aws s3 rm $s3_path_parquet/final_data/ --recursive


In [65]:
# Set S3 staging directory -- this is a temporary directory used for Athena queries
s3_staging_dir = "s3://{0}/athena/staging".format(bucket)

In [66]:
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [67]:
statement = f"""
DROP TABLE IF EXISTS {database_name}.{table_name_parquet};
"""
print(statement)


DROP TABLE IF EXISTS ccpp_aws_fp.final_data_parquet_v1;



In [68]:

df=pd.read_sql(statement, conn)
df

/tmp/ipykernel_18697/750451967.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(statement, conn)


""


In [69]:
# SQL statement to execute
statement = f"""

CREATE TABLE {database_name}.{table_name_parquet}
WITH (
    format = 'PARQUET',
    external_location = 's3://{bucket}/CCPP/data/parquet/final_data_v1/'
) AS
SELECT
    row_number() OVER (ORDER BY gt_energy_yield) AS seq_id,
    CAST(current_timestamp AS timestamp) AS ingestion_ts,
    gt_comp_dis_pressure,
    gt_exhaust_pressure,
    gt_inlet_temp,
    gt_air_filter_diff_pressure,
    solar_radiation,
    solar_energy,
    uv_index,
    gt_ambient_pressure,
    cloud_cover,
    sea_level_pressure,
    gt_energy_yield
FROM {database_name}.{table_name_csv};"""

print(statement)



CREATE TABLE ccpp_aws_fp.final_data_parquet_v1
WITH (
    format = 'PARQUET',
    external_location = 's3://sagemaker-us-east-1-483525480215/CCPP/data/parquet/final_data_v1/'
) AS
SELECT
    row_number() OVER (ORDER BY gt_energy_yield) AS seq_id,
    CAST(current_timestamp AS timestamp) AS ingestion_ts,
    gt_comp_dis_pressure,
    gt_exhaust_pressure,
    gt_inlet_temp,
    gt_air_filter_diff_pressure,
    solar_radiation,
    solar_energy,
    uv_index,
    gt_ambient_pressure,
    cloud_cover,
    sea_level_pressure,
    gt_energy_yield
FROM ccpp_aws_fp.final_data_csv;


In [70]:
import pandas as pd

pd.read_sql(statement, conn)

/tmp/ipykernel_18697/3803073958.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


,rows


In [75]:

statement = """SELECT * FROM {}.{}
     LIMIT 10""".format(
    database_name, table_name_parquet
)

print(statement)

SELECT * FROM ccpp_aws_fp.final_data_parquet_v1
     LIMIT 10


In [76]:
df = pd.read_sql(statement, conn)
df

/tmp/ipykernel_18697/8222894.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


,seq_id,ingestion_ts,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure,gt_energy_yield
0,1,2026-02-04 06:51:25.215,10.0720,18.132,1027.8,2.6356,346.2,1.2,3.0,1024.7,24.7,1027.3,100.02
1,2,2026-02-04 06:51:25.215,10.3860,19.480,1006.5,3.6384,33.8,0.1,0.0,1017.7,24.2,1028.0,100.03
2,3,2026-02-04 06:51:25.215,9.9408,17.982,1037.5,2.5758,761.2,2.7,8.0,1013.5,43.4,1017.8,100.04
3,4,2026-02-04 06:51:25.215,10.0930,18.091,1028.7,2.6372,543.4,2.0,5.0,1023.3,27.8,1027.1,100.07
4,5,2026-02-04 06:51:25.215,10.0720,18.232,1026.9,3.8423,0.0,0.0,0.0,1013.7,45.0,1021.9,100.14
5,6,2026-02-04 06:51:25.215,9.9813,17.761,1038.7,2.7893,706.2,2.5,7.0,1014.4,48.2,1016.7,100.20
6,7,2026-02-04 06:51:25.215,9.9399,17.719,1031.5,2.6382,0.0,0.0,0.0,1010.6,0.0,1013.9,100.20
7,8,2026-02-04 06:51:25.215,9.8518,17.939,1033.0,2.9383,0.0,0.0,0.0,1008.0,46.0,1012.1,100.22
8,9,2026-02-04 06:51:25.215,10.0710,18.039,1030.6,2.6063,487.6,1.8,5.0,1023.9,24.0,1027.2,100.36
9,10,2026-02-04 06:51:25.215,10.2140,18.011,1022.6,2.5765,30.5,0.1,0.0,1021.3,89.1,1009.2,100.38


In [77]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>